This notebook is for exploring short term plasticity



## Relevant papers

- Tsodyks1997_NeuralCode : original paper
- Tsodyks1998_NeuralNetworks : this paper discusses a network 

## Relevant math

## ToDos

- visualize neuron with one synapse

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy

from pathlib import Path
from IPython.display import Image, display
import ipywidgets as widgets  # interactive display

import pyNN.nest as sim
import nest

In [ ]:
def short_term_plasticity(rate, u, tau_rec, tau_fac):
    "Returns stationary limit of short-term plasticity."
    # in tvb:
    #   rates [kHz]
    #   tau_rec [ms]
    #   tau_fac [ms]
    #   u [0, 1]

    # Reshaping to force the ndim=1, shape = (1,)
    u = u.reshape(1)
    tau_rec = tau_rec.reshape(1)
    tau_fac = tau_fac.reshape(1)

    flat_rate = rate.flatten()
    u_steady = np.ones_like(flat_rate) * u
    x_steady = np.ones_like(flat_rate)
    mask = flat_rate >= 1e-9

    if tau_fac != 0:
        exp = np.exp(-1/(flat_rate[mask]*tau_fac))
        u_steady[mask] = u / (1 - (1-u)*exp)

    if tau_rec != 0:
        exp = np.exp(-1/(flat_rate[mask]*tau_rec))
        x_steady[mask] = (1-exp) / (1 - (1-u_steady[mask])*exp)

    flat_stp = u_steady * x_steady
    return flat_stp.reshape(rate.shape)

sim.setup(timestep=0.1)
neuron = sim.Population(1, sim.EIF_cond_exp_isfa_ista())

input_rate = 50  # Hz
simulation_time = 1000  # ms
input_spikes = np.arange(0, simulation_time, 1000/input_rate)[1:]
poisson = False

syn_pars = {'U': 0.03, 'delay': 1.0, 'tau_fac': 530.0, 'tau_rec': 130., 'weight': 20.0}
# syn_pars = {'U': 0.75, 'delay': 1.0, 'tau_fac': 0.0, 'tau_rec': 100., 'weight': 20.0, 'tau_psc': 300.0,}
# syn_pars = {'weight': 0.18, 'delay': 1.0, 'U': 0.75, 'tau_rec': 30, 'tau_psc': 3.0, 'tau_fac': 0.0}
synapse = sim.native_synapse_type('tsodyks_synapse')(**syn_pars)


connector = sim.AllToAllConnector()

if poisson:
    poisson_input = sim.Population(1, sim.SpikeSourcePoisson(rate=input_rate))
    sim.Projection(poisson_input, neuron, connector, synapse_type=synapse)
else:
    regular_input = sim.Population(1, sim.SpikeSourceArray(spike_times=input_spikes))
    sim.Projection(regular_input, neuron, connector, synapse_type=synapse)



neuron.record(['v', 'spikes', 'w', 'gsyn_exc', 'gsyn_inh'])
sim.run(simulation_time)

membrane_potential = neuron.get_data().segments[0].filter(name="v")[0]
spike_times = neuron.get_data().segments[0].spiketrains[0]
adaptation = neuron.get_data().segments[0].filter(name="w")[0]
conductance = neuron.get_data().segments[0].filter(name="gsyn_exc")[0]
sim.end()

# e = np.exp(-1/(input_rate*syn_pars['tau_rec']*1e-3))
# u = syn_pars['U']

stationary_cond = syn_pars['weight'] * 1e-3 * short_term_plasticity(np.array([input_rate])*1e-3, 
                                                                    np.array([syn_pars['U']]), 
                                                                    np.array([syn_pars['tau_rec']]), 
                                                                    np.array([syn_pars['tau_fac']]))


plt.figure(figsize=(10, 6))

plt.subplot(3, 1, 1)
plt.plot(membrane_potential.times, membrane_potential)
plt.ylabel("Membrane potential (mV)")

plt.subplot(3, 1, 2)
plt.plot(conductance.times, conductance)
plt.xlabel("Time (ms)")
plt.ylabel("conductance")
plt.axhline(y=stationary_cond, color='r', linestyle='--')

plt.subplot(3, 1, 3)
plt.plot(spike_times, [0]*len(spike_times), 'ro')  # Plot spike times as red dots
plt.xlim(0, simulation_time)
plt.ylabel("Spikes")

plt.tight_layout()
plt.show()


In [ ]:
sim.setup(timestep=0.1)
input_rate = 500  # Hz
syn_pars = {'U': 0.75, 'delay': 1.0, 'tau_fac': 0.0, 'tau_rec': 1., 'weight': 20.0}
# syn_pars = {'U': 0.75, 'delay': 1.0, 'tau_fac': 0.0, 'tau_rec': 1., 'weight': 20.0, 'tau_psc': 30.0,}
# syn_pars = {'weight': 0.18, 'delay': 1.0, 'U': 0.75, 'tau_rec': 30, 'tau_psc': 3.0, 'tau_fac': 0.0}
synapse = sim.native_synapse_type('tsodyks2_synapse')(**syn_pars)

pulse = sim.DCSource(amplitude=1, start=20.0, stop=800.0)


neuron1 = sim.Population(1, sim.EIF_cond_exp_isfa_ista())
neuron2 = sim.Population(1, sim.EIF_cond_exp_isfa_ista())

connector = sim.AllToAllConnector()

projection = sim.Projection(neuron1, neuron2, connector, synapse_type=synapse, receptor_type='excitatory')
pulse.inject_into(neuron1)


neuron2.record(['v', 'spikes', 'w', 'gsyn_exc', 'gsyn_inh'])
sim.run(simulation_time)

membrane_potential = neuron2.get_data().segments[0].filter(name="v")[0]
spike_times = neuron2.get_data().segments[0].spiketrains[0]
adaptation = neuron2.get_data().segments[0].filter(name="w")[0]
conductance = neuron2.get_data().segments[0].filter(name="gsyn_exc")[0]
# sim.end()

e = np.exp(-1/(input_rate*syn_pars['tau_rec']*1e-3))
u = syn_pars['U']
stationary_cond = u*(1-e)/(1-(1-u)*e) * syn_pars['weight'] *1e-3

plt.figure(figsize=(10, 6))

plt.subplot(3, 1, 1)
plt.plot(membrane_potential.times, membrane_potential)
plt.ylabel("Membrane potential (mV)")

plt.subplot(3, 1, 2)
plt.plot(conductance.times, conductance)
plt.xlabel("Time (ms)")
plt.ylabel("conductance")
plt.axhline(y=stationary_cond, color='r', linestyle='--')

plt.subplot(3, 1, 3)
plt.plot(spike_times, [0]*len(spike_times), 'ro')  # Plot spike times as red dots
plt.xlim(0, simulation_time)
plt.ylabel("Spikes")

plt.tight_layout()
plt.show()


In [ ]:
# Comparison of the two synapses
import pyNN.nest as sim

syn_pars = {'U': 0.75, 'delay': 1.0, 'tau_fac': 0.0, 'tau_rec': 100., 'weight': 20.0}
neuron_model = sim.EIF_cond_exp_isfa_ista()
input_rate = 20  # Hz
simulation_time = 1000  # ms
poisson = False
static = False

##############
# SIMULATION #
##############

tsodyks_params = syn_pars.copy()
tsodyks_synapse = sim.native_synapse_type('tsodyks_synapse')(**tsodyks_params)

tsodyks2_params = syn_pars.copy()
tsodyks2_params['u'] = tsodyks2_params['U']
# tsodyks2_params['U'] = 0.0
tsodyks2_synapse = sim.native_synapse_type('tsodyks2_synapse')(**tsodyks2_params)

static_params = {'weight' : syn_pars['weight'], 'delay' : syn_pars['delay']}
static_synapse = sim.native_synapse_type('static_synapse')(**static_params)

sim.setup(timestep=0.1)

neuron1 = sim.Population(1, neuron_model)
neuron2 = sim.Population(1, neuron_model)
neuron3 = sim.Population(1, neuron_model)

connector = sim.AllToAllConnector()


input_spikes1 = np.arange(0, 300, 1000/input_rate)[1:]
input_spikes2 = np.arange(700, simulation_time, 1000/input_rate)
input_spikes = np.append(input_spikes1, input_spikes2)
if poisson:
    source_neuron = sim.Population(1, sim.SpikeSourcePoisson(rate=input_rate))
else:
    source_neuron = sim.Population(1, sim.SpikeSourceArray(spike_times=input_spikes))
# source_neuron = sim.Population(1, neuron_model)
# dc_source = sim.DCSource(amplitude=1.0, start=10.0, stop=simulation_time)
# source_neuron.inject(dc_source)

projection1 = sim.Projection(source_neuron, neuron1, connector, synapse_type=tsodyks_synapse, receptor_type='excitatory')
projection2 = sim.Projection(source_neuron, neuron2, connector, synapse_type=tsodyks2_synapse, receptor_type='excitatory')
projection3 = sim.Projection(source_neuron, neuron3, connector, synapse_type=static_synapse, receptor_type='excitatory')

neuron1.record(['v', 'spikes', 'w', 'gsyn_exc', 'gsyn_inh'])
neuron2.record(['v', 'spikes', 'w', 'gsyn_exc', 'gsyn_inh'])
neuron3.record(['v', 'spikes', 'w', 'gsyn_exc', 'gsyn_inh'])

sim.run(simulation_time)

v_ms = []
conds = []
if static:
    ns = [neuron1, neuron2, neuron3]
    labels = ['tsodyks_synapse', 'tsodyks2_synapse', 'static_synapse']
else: 
    ns = [neuron1, neuron2]
    labels = ['tsodyks_synapse', 'tsodyks2_synapse']

for n in ns:
    v_ms.append(n.get_data().segments[0].filter(name="v")[0])
    conds.append(n.get_data().segments[0].filter(name="gsyn_exc")[0])

sim.end()

############
# PLOTTING #
############

e = np.exp(-1/(input_rate*syn_pars['tau_rec']*1e-3))
u = syn_pars['U']
stationary_cond = u*(1-e)/(1-(1-u)*e) * syn_pars['weight'] *1e-3

# w = syn_pars['weight']
# u = syn_pars['U']
# tau = syn_pars['tau_rec']
peaks = [syn_pars['weight']*syn_pars['U']]
for dt in np.diff(input_spikes):
    peak = peaks[-1]*(1-syn_pars['U'])*np.exp(-dt/syn_pars['tau_rec']) + syn_pars['weight']*syn_pars['U']*(1-np.exp(-dt/syn_pars['tau_rec']))
    peaks.append(peak)
peaks = np.array(peaks)*1e-3

plt.figure(figsize=(10, 6))
plt.subplot(2, 1, 1)
for membrane_potential, label in zip(v_ms, labels):
    plt.plot(membrane_potential.times, membrane_potential, label=label)
plt.ylabel("Membrane potential (mV)")
plt.legend()

plt.subplot(2, 1, 2)
for conductance, label in zip(conds, labels):
    plt.plot(conductance.times, conductance, label=label)
plt.plot(input_spikes+1, peaks, "o", label='theory')
plt.xlabel("Time (ms)")
plt.ylabel("conductance")
plt.axhline(y=stationary_cond, color='r', linestyle='--', label='stationary')
plt.legend()

# plt.subplot(3, 1, 3)
# plt.plot(spike_times, [0]*len(spike_times), 'ro')  # Plot spike times as red dots
# plt.xlim(0, simulation_time)
# plt.ylabel("Spikes")


plt.tight_layout()
plt.show()


In [ ]:
print("tsodyks_synapse")
for p in ['U', 'tau_psc', 'tau_rec', 'tau_fac', 'x', 'y', 'u', 'delay', 'weight']:
    print(f"{p} : {projection1.get(p, format='list')[0][-1]}")
print("")
print("tsodyks2_synapse")
for p in ['U', 'tau_rec', 'tau_fac', 'x', 'u', 'delay', 'weight']:
    print(f"{p} : {projection2.get(p, format='list')[0][-1]}")

# projection1.get('tau_rec', format='list')[0][-1]

In [ ]:
plt.plot(input_spikes, peaks, "o")
plt.show()

# NEST

In [ ]:
import matplotlib.pyplot as plt
import nest
import nest.voltage_trace

nest.ResetKernel()

dep_params = {"U": 0.67, "u": 0.67, "x": 1.0, "tau_rec": 450.0, "tau_fac": 0.0, "weight": 250.0}
fac_params = {"U": 0.1, "u": 0.1, "x": 1.0, "tau_fac": 1000.0, "tau_rec": 100.0, "weight": 250.0}

# tsodyks_params = dict(fac_params, synapse_model="tsodyks_synapse")  # for tsodyks_synapse
# tsodyks2_params = dict(fac_params, synapse_model="tsodyks2_synapse")  # for tsodyks2_synapse
tsodyks_params = dict(dep_params, synapse_model="tsodyks_synapse")  # for tsodyks_synapse
tsodyks2_params = dict(dep_params, synapse_model="tsodyks2_synapse")  # for tsodyks2_synapse

neuron = nest.Create("iaf_psc_exp", 3, params={"tau_syn_ex": 3.0})
# multimeter = nest.Create("multimeter", params={"interval": 0.1, "record_from": ["g_ex", "g_in"]})


nest.Connect(neuron[0], neuron[1], syn_spec=tsodyks_params)
nest.Connect(neuron[0], neuron[2], syn_spec=tsodyks2_params)

voltmeter = nest.Create("voltmeter", 2)
nest.Connect(voltmeter[0], neuron[1])
nest.Connect(voltmeter[1], neuron[2])

mm = nest.Create("multimeter", 2, params={"record_from": ['I_syn_ex']})
nest.Connect(mm[0], neuron[1])
nest.Connect(mm[1], neuron[2])


neuron[0].I_e = 376.0
nest.Simulate(500.0)
neuron[0].I_e = 0.0
nest.Simulate(1000.0)
neuron[0].I_e = 376.0
nest.Simulate(500.0)

nest.voltage_trace.from_device(voltmeter[0])
nest.voltage_trace.from_device(voltmeter[1])
plt.show()

events = nest.GetStatus(mm[0], "events")[0]
times = events["times"]
I_syn_ex = events["I_syn_ex"]
plt.plot(times[:100], I_syn_ex[:100], label="Neuron2")
events = nest.GetStatus(mm[1], "events")[0]
times = events["times"]
I_syn_ex = events["I_syn_ex"]
plt.plot(times[:100], I_syn_ex[:100], label="Neuron2")
plt.show()




In [ ]:
import matplotlib.pyplot as plt
import nest
import nest.voltage_trace

neurons = ["iaf_psc_exp", "iaf_psc_alpha", "iaf_cond_exp", "aeif_cond_alpha", "aeif_cond_exp", "aeif_psc_alpha"]

for neuron_name in neurons:
    nest.ResetKernel()
    print(neuron_name)
    if 'cond' in neuron_name:
        record = "g_ex"
    elif 'psc' in neuron_name:
        record = 'I_syn_ex'

    # depression
    syn_pars = {"U": 0.67, "u": 0.67, "x": 1.0, "tau_rec": 450.0, "tau_fac": 0.0, "weight": 250.0}
    # facilitation
    # syn_params = {"U": 0.1, "u": 0.1, "x": 1.0, "tau_fac": 1000.0, "tau_rec": 100.0, "weight": 250.0}

    wr1 = nest.Create("weight_recorder")
    wr2 = nest.Create("weight_recorder")
    nest.CopyModel("tsodyks_synapse", "tsodyks_synapse_rec", {"weight_recorder": wr1})
    nest.CopyModel("tsodyks2_synapse", "tsodyks2_synapse_rec", {"weight_recorder": wr2})
    
    tsodyks_params = dict(syn_pars, synapse_model="tsodyks_synapse_rec")  # for tsodyks_synapse
    # tsodyks_params['tau_psc'] = 200.0
    tsodyks2_params = dict(syn_pars, synapse_model="tsodyks2_synapse_rec")  # for tsodyks2_synapse
    
    source = nest.Create('iaf_psc_exp', 1, params={"tau_syn_ex": 3.0})
    neuron = nest.Create(neuron_name, 2, params={"tau_syn_ex": 3.0})
    
    nest.Connect(source, neuron[0], syn_spec=tsodyks_params)
    nest.Connect(source, neuron[1], syn_spec=tsodyks2_params)
    
    voltmeter = nest.Create("voltmeter", 2)
    nest.Connect(voltmeter[0], neuron[0])
    nest.Connect(voltmeter[1], neuron[1])
    
    mm = nest.Create("multimeter", 2, params={"record_from": [record]})
    nest.Connect(mm[0], neuron[0])
    nest.Connect(mm[1], neuron[1])
    
    source.I_e = 376.0
    nest.Simulate(500.0)
    source.I_e = 0.0
    nest.Simulate(1000.0)
    source.I_e = 376.0
    nest.Simulate(500.0)
    
    nest.voltage_trace.from_device(voltmeter[0])
    nest.voltage_trace.from_device(voltmeter[1])
    plt.show()

    st = 1550
    en = 1650
    
    events = nest.GetStatus(mm[0], "events")[0]
    times = events["times"]
    syn_input = events[record]
    plt.plot(times[st:en], syn_input[st:en], label="Neuron2")

    events = nest.GetStatus(mm[1], "events")[0]
    times = events["times"]
    syn_input = events[record]
    plt.plot(times[st:en], syn_input[st:en], label="Neuron3")

    spike_times = wr1.get('events', 'times')
    u = syn_pars['U']
    w = syn_pars['weight']
    tau = syn_pars['tau_rec']
    peaks = [w*u]
    for dt in np.diff(spike_times):
        peak = peaks[-1]*(1-u)*np.exp(-dt/tau) + w*u*(1-np.exp(-dt/tau))
        peaks.append(peak)
    peaks = np.array(peaks)
    
    theory = np.zeros_like(times)
    tau = neuron[1].get('tau_syn_ex')
    # tau = tsodyks_params['tau_psc']
    if 'alpha' in neuron_name:
        for spike, w  in zip(spike_times, peaks):
            instance = w*np.exp(-(times-spike)/tau)*(times-spike)/(tau)*np.exp(1)
            instance[times<spike] = 0
            theory += instance
    elif 'exp' in neuron_name:
        for spike, w  in zip(spike_times, peaks):
            instance = w*np.exp(-(times-spike)/tau)
            instance[times<spike] = 0
            theory += instance
        
    plt.plot((times+1)[st:en], theory[st:en],"--",  label="Theory")
    # compute theoretical conductance shape 
    
    plt.ylabel(record)
    plt.legend()
    plt.show()

In [ ]:
# -*- coding: utf-8 -*-
#
# evaluate_tsodyks2_synapse.py
#
# This file is part of NEST.
#
# Copyright (C) 2004 The NEST Initiative
#
# NEST is free software: you can redistribute it and/or modify
# it under the terms of the GNU General Public License as published by
# the Free Software Foundation, either version 2 of the License, or
# (at your option) any later version.
#
# NEST is distributed in the hope that it will be useful,
# but WITHOUT ANY WARRANTY; without even the implied warranty of
# MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
# GNU General Public License for more details.
#
# You should have received a copy of the GNU General Public License
# along with NEST.  If not, see <http://www.gnu.org/licenses/>.

"""
Example of the tsodyks2_synapse in NEST
---------------------------------------

This synapse model implements synaptic short-term depression and short-term f
according to [1]_ and [2]_. It solves Eq (2) from [1]_ and modulates U according

This connection merely scales the synaptic weight, based on the spike history
parameters of the kinetic model. Thus, it is suitable for any type of synapse
that is current or conductance based.

The parameter `A_se` from the publications is represented by the
synaptic weight. The variable `x` in the synapse properties is the
factor that scales the synaptic weight.

See also [3]_.

Parameters
~~~~~~~~~~

The following parameters can be set in the status dictionary:

* U           - probability of release increment (U1) [0,1], default=0.
* u           - Maximum probability of release (U_se) [0,1], default=0.
* x           - current scaling factor of the weight, default=U
* tau_rec     - time constant for depression in ms, default=800 ms
* tau_fac     - time constant for facilitation in ms, default=0 (off)

Notes
~~~~~

Under identical conditions, the ``tsodyks2_synapse`` produces slightly lower
peak amplitudes than the ``tsodyks_synapse``. However, the qualitative behavior
is identical.

This compares the two synapse models.

References
~~~~~~~~~~

.. [1] Tsodyks MV, and Markram H. (1997). The neural code between
       neocortical depends on neurotransmitter release probability. PNAS,
       94(2), 719-23.
.. [2] Fuhrmann G, Segev I, Markram H, and Tsodyks MV. (2002). Coding of
       temporal information by activity-dependent synapses. Journal of
       Neurophysiology, 8. https://doi.org/10.1152/jn.00258.2001
.. [3] Maass W, and Markram H. (2002). Synapses as dynamic memory buffers.
       Neural Networks, 15(2), 155-161.
       http://dx.doi.org/10.1016/S0893-6080(01)00144-7
"""

import matplotlib.pyplot as plt
import nest
import nest.voltage_trace

nest.ResetKernel()

###############################################################################
# Parameter set for depression

dep_params = {"U": 0.67, "u": 0.67, "x": 1.0, "tau_rec": 450.0, "tau_fac": 0.0, "weight": 250.0}

###############################################################################
# Parameter set for facilitation

fac_params = {"U": 0.1, "u": 0.1, "x": 1.0, "tau_fac": 1000.0, "tau_rec": 100.0, "weight": 250.0}

###############################################################################
# Now we assign the parameter set to the synapse models.

tsodyks_params = dict(fac_params, synapse_model="tsodyks_synapse")  # for tsodyks_synapse
tsodyks2_params = dict(fac_params, synapse_model="tsodyks2_synapse")  # for tsodyks2_synapse

###############################################################################
# Create three neurons.

neuron = nest.Create("iaf_psc_exp", 3, params={"tau_syn_ex": 3.0})

###############################################################################
# Neuron one produces spikes. Neurons 2 and 3 receive the spikes via the two
# synapse models.

nest.Connect(neuron[0], neuron[1], syn_spec=tsodyks_params)
nest.Connect(neuron[0], neuron[2], syn_spec=tsodyks2_params)

###############################################################################
# Now create two voltmeters to record the responses.

voltmeter = nest.Create("voltmeter", 2)

###############################################################################
# Connect the voltmeters to the neurons.

nest.Connect(voltmeter[0], neuron[1])
nest.Connect(voltmeter[1], neuron[2])

###############################################################################
# Now simulate the standard STP protocol: a burst of spikes, followed by a
# pause and a recovery response.

neuron[0].I_e = 376.0

nest.Simulate(500.0)
neuron[0].I_e = 0.0
nest.Simulate(500.0)
neuron[0].I_e = 376.0
nest.Simulate(500.0)

###############################################################################
# Finally, generate voltage traces. Both are shown in the same plot and
# should be almost completely overlapping.

nest.voltage_trace.from_device(voltmeter[0])
nest.voltage_trace.from_device(voltmeter[1])
plt.show()